In [ ]:
import os

# 1. 强制清理旧残留
print("正在清理旧文件...")
!rm -rf Diffusion-Illusions
!rm -rf Diffusion-Illusion
!rm -rf master.zip
!rm -rf generated_qr_outputs # 清理旧的输出

# 2. 克隆仓库
print("正在克隆仓库...")
!git clone https://github.com/RyannDaGreat/Diffusion-Illusions

# 3. 检查是否成功
if os.path.exists('Diffusion-Illusions'):
    print("✅ 仓库克隆成功！")

    # 4. 进入目录
    %cd Diffusion-Illusions

    # 5. 安装依赖
    print("正在安装依赖 (包括二维码生成库 amzqr)...")
    !pip install -r requirements.txt
    !pip install mediapy easydict "numpy<2.0" amzqr qrcode[pil]

    print("\n✅✅ 环境初始化全部完成！")
    print("⚠️⚠️ 现在的关键步骤：请点击上方菜单 'Runtime' -> 'Restart session' 重启运行时！")

else:
    print("❌❌ 克隆还是失败了，请检查网络。")

In [ ]:
import os

# 定义仓库名字
repo_name = "Diffusion-Illusions"

# 检查当前是否已经在文件夹里了
if os.getcwd().endswith(repo_name):
    print(f"✅ 当前位置正确: {os.getcwd()}")
else:
    # 如果不在，就尝试进去
    if os.path.exists(repo_name):
        %cd {repo_name}
        print(f"✅ 已切换工作目录到: {os.getcwd()}")
    else:
        print("⚠️ 文件夹不存在，正在重新克隆...")
        !git clone https://github.com/RyannDaGreat/Diffusion-Illusions
        %cd {repo_name}
        print(f"✅ 克隆并切换完成: {os.getcwd()}")

In [ ]:
from rp import *
import numpy as np
import rp
import torch
import torch.nn as nn
import source.stable_diffusion as sd
from source.learnable_textures import LearnableImageFourier
from source.stable_diffusion_labels import NegativeLabel
from itertools import chain
import torchvision.transforms.functional as TF
from torchvision import transforms
from PIL import Image, ImageOps

# === 核心工具函数 (已重构) ===

def load_processed_tensor(file_path, target_size=(256, 256)):
    """
    [重构] 读取图片，缩放并转为 Tensor。
    原函数名: read_and_convert_image_to_tensor
    """
    # 1. 打开并转 RGB
    pil_img = Image.open(file_path).convert("RGB")

    # 2. 定义转换管道 (显式增加了 LANCZOS 插值，提升缩放质量)
    transform_pipeline = transforms.Compose([
        transforms.Resize(target_size, interpolation=transforms.InterpolationMode.LANCZOS),
        transforms.ToTensor()
    ])

    # 3. 处理并移动到 GPU
    return transform_pipeline(pil_img).to(device)

def blend_with_background(img_tensor, opacity=0.5, bg_color=1.0):
    """
    [重构] 将图片与纯色背景混合，降低对比度。
    原函数名: fade_tensor_image

    Args:
        img_tensor: 输入图像
        opacity: 二维码的不透明度 (alpha)，越小越隐蔽
        bg_color: 背景颜色值 (1.0 代表纯白)
    """
    # 确保是浮点数
    x = img_tensor.float()

    # 自动归一化检测 (防止输入是 0-255 范围)
    if x.max() > 1.0:
        x = x / 255.0

    # 混合公式: opacity * 图 + (1 - opacity) * 背景
    # 逻辑与原版一致，写法更直观
    mixed_result = x * opacity + bg_color * (1.0 - opacity)

    return mixed_result

In [ ]:
# 初始化 GPU 和 模型
if 'model_sd' not in dir():
    print("正在加载 Stable Diffusion...")
    model_name = "CompVis/stable-diffusion-v1-4"
    gpu = rp.select_torch_device()
    model_sd = sd.StableDiffusion(gpu, model_name)
    device = model_sd.device
    print("模型加载完毕！")
else:
    print("模型已存在，跳过加载。")

In [ ]:
# === Cell 5 (修正版): 强制上传并读取新的黑白二维码 ===
from google.colab import files
import cv2
import os

print(">>> 请点击下方按钮，上传你的黑白二维码图片 (QRCode.png) <<<")

# 1. 强制弹出上传框
uploaded = files.upload()

if uploaded:
    # 2. 获取你刚刚上传的这个文件的名字 (而不是去文件夹里乱找)
    filename = next(iter(uploaded))
    print(f"正在处理你上传的图片: {filename} ...")

    # 3. 读取并转为灰度
    img_cv = cv2.imread(filename, cv2.IMREAD_GRAYSCALE)

    if img_cv is None:
        print(f"❌ 错误：无法读取 {filename}，请确保上传的是有效的图片文件。")
    else:
        # 4. 强力二值化 (Thresholding)
        # 设定阈值为 128，强制把图片变成非黑即白
        _, img_binary = cv2.threshold(img_cv, 128, 255, cv2.THRESH_BINARY)

        # 5. 转回 PIL 并缩放
        target_pil = Image.fromarray(img_binary).convert('RGB')
        target_pil = ImageOps.fit(target_pil, (256, 256), method=Image.Resampling.LANCZOS)

        # 6. 转 Tensor
        raw_qr_tensor = TF.to_tensor(target_pil).to(device)

        # 7. 生成遮罩 (Target Mask)
        # 黑色区域 (必须变暗)
        mask_black = (raw_qr_tensor < 0.5).float()
        # 白色区域 (必须变亮)
        mask_white = (raw_qr_tensor >= 0.5).float()

        print("\n✅ 新二维码处理完毕！")
        print("Mask Black (需变黑区域) 预览:")
        rp.display_image(rp.as_numpy_image(mask_black))

else:
    print("❌ 你取消了上传，请重新运行此代码块！")

In [ ]:
# === Cell 6: 重新初始化 (乘法叠加版) ===

class RelaxedQuadSolver:
    def __init__(self, config, device_ref):
        self.device = device_ref
        
        # 直接使用全局变量里的 mask (来自 Cell 5)
        self.is_black_target = mask_black
        self.is_white_target = mask_white
        
        self.prompts = [
            config['layer_1'], config['layer_2'], 
            config['layer_3'], config['layer_4']
        ]
        self.neg_prompt = config.get('negative', "")
        
        self.generators = [self._create_generator() for _ in range(4)]
        self._setup_optimization()

    def _create_generator(self):
        # 初始化生成器
        return LearnableImageFourier(height=256, width=256, hidden_dim=256, num_features=128).to(self.device)

    def _get_mixed_image(self):
        # === 修改点：乘法叠加 (Multiply) ===
        # 逻辑：Result = Image1 * Image2 * Image3 * Image4
        # 这种模式下，为了让最终结果是白色的，所有的子图都必须非常亮（接近白色）。
        # 这就是为什么“减色法”生成的图片通常看起来很亮/很淡的原因。
        
        mixed = None
        for g in self.generators:
            img = g()
            # 关键：确保像素值在 0-1 之间，否则乘法会乱套
            # 使用 clamp 限制范围
            img = torch.clamp(img, 0.0, 1.0)
            
            if mixed is None:
                mixed = img
            else:
                mixed = mixed * img
                
        return mixed

    def _setup_optimization(self):
        self.labels = [NegativeLabel(p, self.neg_prompt) for p in self.prompts]
        all_params = chain(*[g.parameters() for g in self.generators])
        self.optimizer = torch.optim.SGD(all_params, lr=1e-4)

    def get_training_components(self):
        gen_funcs = [lambda g=g: g() for g in self.generators]
        func_mix = lambda: self._get_mixed_image()
        return self.labels, gen_funcs, self.optimizer, func_mix

# --- 🚀 配置 ---
user_config = {
    # 提示词保持不变，但由于乘法会让图片变暗，
    # 建议提示词里多加一些 bright, white background 相关的词
    'layer_1': "a giraffe face, high contrast, realistic, white background",
    'layer_2': "Hatsune Miku, anime style, vivid colors, white background",
    'layer_3': "a green frog, bright lighting, realistic, white background",
    'layer_4': "a cat face, high contrast, fluffy, cute, white background",
    'negative': "blur, gray, low contrast, ugly, text, watermark, qr code, dark, black background"
}

print("正在重新初始化 (乘法模式)...")
solver = RelaxedQuadSolver(user_config, device)
train_labels, train_images, optim, get_mixed_image = solver.get_training_components()
print("✅ 完成。准备开始乘法逻辑训练。")

In [ ]:
# === Cell 7: 训练循环 (适配乘法版) ===
import torch.nn.functional as F

NUM_ITER = 3000
DISPLAY_INTERVAL = 200

# ⚠️ 调整后的参数
# 乘法模式下，保持内容可视性比较难，权重适当调整
STYLE_GUIDANCE = 80   
QR_GUIDANCE = 8000    # 加大二维码权重，因为乘法要这就得 4 张图同时配合，难度更大

display_eta = rp.eta(NUM_ITER, title='Training Status')

print(f"🚀 开始训练 (乘法模式)... (Target: 权重 {QR_GUIDANCE})")

try:
    for iter_num in range(NUM_ITER):
        display_eta(iter_num)

        # 1. Style Loss (画图)
        weights = rp.as_numpy_array([1, 1, 1, 1])
        weights = weights / weights.sum() * len(weights)
        batch = rp.random_batch(list(zip(train_labels, train_images, weights)), batch_size=1)

        for label, get_image_func, weight in batch:
            _ = model_sd.train_step(
                label.embedding,
                get_image_func()[None],
                noise_coef=0.1 * weight,
                guidance_scale=STYLE_GUIDANCE,
            )

        # 2. QR Loss (隐写)
        current_mix = get_mixed_image()

        # 乘法后的结果通常偏暗，所以我们需要用力惩罚那些“不够白”的地方
        
        # 黑区目标：亮度 < 0.3 (保持黑色)
        loss_black = torch.mean( (F.relu(current_mix - 0.3) * mask_black) ** 2 )

        # 白区目标：亮度 > 0.7 (保持白色)
        # 在乘法里，这很难，因为 0.9*0.9*0.9*0.9 = 0.65
        # 所以每张图必须达到 0.92 以上才能让结果 > 0.7
        loss_white = torch.mean( (F.relu(0.7 - current_mix) * mask_white) ** 2 )

        loss_secret = (loss_black + loss_white) * QR_GUIDANCE

        loss_secret.backward()

        # 3. 显示
        with torch.no_grad():
            if iter_num % DISPLAY_INTERVAL == 0:
                from IPython.display import clear_output
                clear_output(wait=True)

                imgs = [rp.as_numpy_image(f()) for f in train_images]
                img_mix = rp.as_numpy_image(get_mixed_image())

                print(f"Iteration {iter_num} / {NUM_ITER}")
                print("上排: 长颈鹿 | 初音 | 青蛙")
                print("下排: 猫咪   | 【乘法叠加结果】(应为白底黑码) | (Target Mask)")

                row1 = np.hstack([imgs[0], imgs[1], imgs[2]])
                target_vis = rp.as_numpy_image(mask_black)
                row2 = np.hstack([imgs[3], img_mix, target_vis])

                rp.display_image(np.vstack([row1, row2]))

        optim.step()
        optim.zero_grad()

except KeyboardInterrupt:
    print("停止训练")

# 保存
print("保存结果...")
for i, f in enumerate(train_images):
    # 保存时确保也是 RGB 格式
    rp.save_image(rp.as_numpy_image(f()), f"layer_{i+1}.png")
rp.save_image(rp.as_numpy_image(get_mixed_image()), "final_multiplied_qr.png")
print("✅ 图片已保存 (乘法模式)")